# Cerberus: PII out before the LLM sees it

A local pipeline. It reads PDF, DOCX and PPTX files (scanned pages and pictures included), finds and classifies personal data, replaces it with stable tokens, and checks its own output before releasing anything. No document content leaves this machine.

**How this notebook is organised**

| Part | What happens | Data |
|---|---|---|
| 1 | The whole pipeline runs live, start to finish | Three small made-up files (about a minute) |
| 2 | Results of the same pipeline on the full documents | A run made beforehand, read from disk |
| 3 | How well it works: recall measured against hand labels | Evaluation files, read from disk |

Part 1 proves it runs. Parts 2 and 3 show what it does on real documents without waiting for OCR.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")            # the repository root
warnings.filterwarnings("ignore")

from pathlib import Path
from IPython.display import HTML, Markdown, display
from cerberus import demo
from cerberus.eval import synthetic

# Where things are. The sample files are made up and safe to show.
SAMPLES  = Path("../outputs/demo/input")    # part 1: files to process live
LIVE_OUT = Path("../outputs/demo/run")      # part 1: where their outputs go
CASE_IN  = Path("../data/case2")            # part 2: the full documents (never committed)
CASE_OUT = Path("../outputs/run3")          # part 2: a finished run on them
EVAL     = Path("../outputs/eval")          # part 3: evaluation results

## Part 1. The pipeline, live

### 1.1 The input

Three files, one per format, written by `cerberus.eval.synthetic`. Every name and number in them is invented. They reproduce the hard cases by *format*:

- **`incident_scan.pdf`**: a scanned page with no text layer, so everything must be read by OCR. Identifiers sit in ordinary sentences with no field labels.
- **`staff_notice.docx`**: typed text, a table, and a picture of an access card whose text exists only as pixels. The author is stored in the file's hidden properties.
- **`team.pptx`**: separate text boxes stored out of reading order, and a table that uses abbreviated names ("J. Sample").

To run your own files instead, put them in the `SAMPLES` folder and skip the next cell.

In [ ]:
for path in synthetic.make(SAMPLES):
    print(path.name)

### 1.2 Run it

One call does everything:

`extract → detect (rules, a local language model, a name sweep) → resolve to tokens → redact → verify → release`

Outputs are written to a holding folder first. They are released only if the pipeline, re-reading its own output, finds no personal data left in it. Otherwise nothing is released and the run fails.

In [ ]:
result, seconds = demo.run(SAMPLES, LIVE_OUT)
print(f"{len(result.documents)} files, {len(result.tokenized)} PII instances, "
      f"{len({t.token for t in result.tokenized})} distinct identities, {seconds:.0f} s")
print("verification:", "PASSED, outputs released" if not result.problems else "FAILED, outputs withheld")

### 1.3 What was found

Instances by category, and by **where** they were: a labelled field (*body*), a table cell, inside a sentence (*prose*), or inside a picture (*image*). The same person counted once under *Distinct*, however many times and in however many files they appear.

In [ ]:
display(HTML(demo.counts(result.tokenized)))

### 1.4 Every finding is explained

One row per finding: the file, page and location; what kind of data it is; **which detector and rule reported it, and how confident it was**. This is the audit log. The released version carries the token only. The original value is shown here because these documents are made up.

In [ ]:
display(HTML(demo.findings(result.documents, result.tokenized, limit=20, values=True)))

### 1.5 What the LLM receives

Left: the text as extracted, with detected values highlighted. Right: the same text as it would be passed downstream.

Things to notice:

- **One person, one token, in every file.** "Jane Sample" in the DOCX, the PDF and the PPTX is the same token, and so is "J. Sample" in the slide's table. The model can still follow who did what.
- **Structure is kept**: headings, table rows and reading order are unchanged.
- **Field labels are left alone**: "primary supplier contact" is not a person.

In [ ]:
for doc in result.documents:
    display(Markdown(f"**{doc.name}**"))
    display(HTML(demo.text_views(doc, result.tokenized, limit=8)))

### 1.6 The masked files

The original files, same pages and layout, with the data removed (not covered: the pixels and characters are gone from the file).

The scanned page first. Every word on it was read by OCR, and each finding is erased at the position of its words.

In [ ]:
display(HTML(demo.pages(SAMPLES / "incident_scan.pdf", LIVE_OUT / "masked" / "incident_scan.pdf", page=1,
                        clip=(0, 0.05, 1, 0.45), dpi=130)))

The picture inside the Word file. Its text is not in the document at all, only in the image. Two OCR engines read it; where they agree the picture is trusted and masked word by word, and the barcode is found from its pattern. A picture they do not agree on is replaced whole.

In [ ]:
display(HTML(demo.pictures(SAMPLES / "staff_notice.docx", LIVE_OUT / "masked" / "staff_notice.docx", number=1)))

### 1.7 The report

Written on every run: totals, which identities appear in more than one file, which were seen **only** inside a picture, and the result of verification. Tokens and counts only, so it can be handed to an auditor who is not cleared to see the data.

In [ ]:
display(Markdown(demo.saved_report(LIVE_OUT)))

## Part 2. The full documents

The same command, run beforehand on the three case documents (a first run takes about 50 minutes, most of it reading 58 pictures with two OCR engines). Everything below is read from that run's output folder.

Nothing in this part shows an original value except the left-hand side of the before/after pictures.

In [ ]:
if CASE_OUT.exists():
    display(HTML(demo.saved_counts(CASE_OUT)))
    display(Markdown(demo.saved_report(CASE_OUT)))
else:
    print("No finished run at", CASE_OUT, "- run:  python -m cerberus.pipeline data/case2 --out outputs/run3")

### 2.1 One file in detail

Pages read by OCR, what was withheld, and counts by category, location and page.

In [ ]:
if CASE_OUT.exists():
    summary = sorted((CASE_OUT / "report").glob("*.summary.md"))[1]   # 0, 1, 2: choose a file
    display(Markdown(summary.read_text(encoding="utf-8")))

### 2.2 The redacted text

A stretch of what the LLM would be given for the scanned policy document: narrative prose with no field labels, where field-based extraction finds nothing.

In [ ]:
if CASE_OUT.exists():
    stem = sorted((CASE_OUT / "redacted").glob("*.md"))[1].stem
    display(HTML(demo.text_file(CASE_OUT, stem, lines=(0, 30))))    # change the line range to move through the file

### 2.3 Before and after

A page of the scanned document. Change `PAGE` to move through it. Good pages to show: a table of people, a page of narrative prose, a page with an embedded form, the page with the ID badge.

In [ ]:
PAGE = 18
if CASE_OUT.exists():
    name = next(p.name for p in demo.files_in(CASE_IN) if p.suffix == ".pdf")
    display(HTML(demo.pages(CASE_IN / name, CASE_OUT / "masked" / name, page=PAGE)))

## Part 3. How well does it work?

Measured against labels a person made by reading the pages, without looking at the pipeline's output. A label and a finding are paired by page and text; *near* means OCR changed up to two characters; *partial* means one text contains the other.

Two questions are kept apart: was the value **read** at all (extraction), and if it was, was it **detected**?

In [ ]:
print(demo.evaluation(EVAL))

### What the numbers say

- **Typed and scanned text (tables, prose, body): every labelled value was found.**
- **Inside pictures it is about half**, up from about a third with one OCR engine. The values missed are mostly ones no engine could read: print seven pixels tall in a screenshot.
- **That gap does not reach the LLM.** Text read from pictures is never passed downstream, and a picture the two engines disagree on is masked whole.

### What is not done

- Labels cover seven pages by one person. The figures are a first measurement.
- There is no precision figure yet: false positives outside pictures have not been counted.
- A first run on the full documents takes about 50 minutes on a laptop CPU.